# Sesión 5 · Práctica guiada: vuestras funciones de gráficos

Esta práctica la hacemos juntos, paso a paso: el profesor escribe y explica cada celda, y tú la reproduces en tu notebook.

Vais a construir, paso a paso, una función que dibuja la evolución de las hipotecas por mil habitantes de cualquier grupo de provincias, y a preparar los datos de una pequeña aplicación.

In [ ]:
# Preparacion: localiza la carpeta de la asignatura (y la descarga si estas en Google Colab).
# Ejecuta esta celda primero; no hace falta entenderla todavia.
import os, subprocess
from pathlib import Path

REPO = "https://github.com/luisaugustos/ppy-analisis-social.git"


def _buscar_carpeta():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        for candidata in (carpeta, carpeta / "ppy-analisis-social"):
            if (candidata / "datos" / "ORIGEN.md").is_file():
                return candidata
    return None


raiz = _buscar_carpeta()
if raiz is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    raiz = Path("ppy-analisis-social").resolve()

os.chdir(raiz)
print("Carpeta de trabajo lista. Los datos estan en la subcarpeta 'datos'.")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

hipotecas = pd.read_csv("datos/hipotecas.csv", dtype={"cod_prov": str}, parse_dates=["fecha"])
poblacion = pd.read_csv("datos/poblacion.csv", dtype={"cod_prov": str})

## Ejercicio 1 · Un gráfico con figura y ejes

Dibujad la serie mensual de hipotecas de Granada con `fig, ax = plt.subplots()`. Añadid título, etiqueta del eje vertical y haced que ese eje empiece en cero.

In [ ]:
granada = hipotecas[hipotecas["provincia"] == "Granada"]

# TODO 1
...

assert ax.get_title() != "" and ax.get_ylim()[0] == 0

## Ejercicio 2 · Añadir la media móvil

Repetid el gráfico con dos líneas: el dato mensual en gris claro y la media móvil de 12 meses encima, más gruesa. Añadid la leyenda.

In [ ]:
serie = granada.set_index("fecha")["hipotecas"]

# TODO 2
...

assert len(ax.lines) == 2

## Ejercicio 3 · Convertirlo en función

Escribid `grafico_serie(tabla, provincia, ax=None)`, que haga el gráfico del ejercicio 2 para cualquier provincia. Si no recibe `ax`, crea su propia figura. Debe devolver `ax`.

In [ ]:
# TODO 3
...


ax = grafico_serie(hipotecas, "Salamanca")
assert ax.get_title() == "Salamanca"

## Ejercicio 4 · Componer varios gráficos

Usad vuestra función para dibujar, en una sola figura de 2 × 2, cuatro provincias a vuestra elección. Guardad la figura en `resultados/cuatro_provincias.png`.

Pista: `fig, ejes = plt.subplots(2, 2)` devuelve una cuadrícula; `ejes.flat` permite recorrerla en un bucle.

In [ ]:
elegidas = ["Salamanca", "Granada", "Valladolid", "Málaga"]

# TODO 4
...

assert Path("resultados/cuatro_provincias.png").exists()

## Ejercicio 5 · Comparar con tasas

Granada tiene casi el triple de población que Salamanca: comparar totales engaña. Completad la función `tasa_anual`, que devuelve las hipotecas anuales por cada mil habitantes de cada provincia (años completos, usando la última población conocida cuando falte).

In [ ]:
def tasa_anual(hipotecas, poblacion):
    """Hipotecas anuales por mil habitantes de cada provincia (solo años completos)."""
    provincias = hipotecas[hipotecas["cod_prov"] != "00"].copy()
    provincias["anyo"] = provincias["fecha"].dt.year
    anual = (
        provincias
        .groupby(["cod_prov", "provincia", "anyo"], as_index=False)
        .agg(hipotecas=("hipotecas", "sum"), meses=("fecha", "count"))
    )
    anual = anual[anual["meses"] == 12]

    # TODO 5: unid la población (how="left"), rellenad los huecos por provincia con ffill y calculad 'tasa_mil'
    ...
    return anual


tasas = tasa_anual(hipotecas, poblacion)
assert "tasa_mil" in tasas.columns and tasas["tasa_mil"].notna().all()
tasas.tail(3)

## Ejercicio 6 · Una función para comparar provincias

Escribid `grafico_comparacion(tasas, provincias)`, que dibuje en un mismo gráfico una línea por provincia con su `tasa_mil` anual, con leyenda, título y el eje vertical desde cero.

In [ ]:
# TODO 6
...


ax = grafico_comparacion(tasas, ["Salamanca", "Granada", "Soria"])
assert len(ax.lines) == 3

## Ejercicio 7 · Versión interactiva

Haced el mismo gráfico con Plotly: `px.line` con `x="anyo"`, `y="tasa_mil"` y `color="provincia"`.

In [ ]:
import plotly.express as px

# TODO 7
...

## Ejercicio 8 · Vuestra primera aplicación (fuera de Colab)

En vuestro ordenador:

1. Copiad `apps/app_plantilla.py` como `apps/app_grupo.py`.
2. Añadid un control `st.sidebar.radio` para elegir entre «Hipotecas al mes» e «Hipotecas por mil habitantes».
3. Usad vuestra función `tasa_anual` para la segunda opción.
4. Ejecutad `uv run streamlit run apps/app_grupo.py`.

## Para cerrar

- ¿Cuántas líneas habríais escrito sin funciones para hacer los gráficos de los ejercicios 4 y 6?
- **Debate (SE4):** un gráfico sobre vivienda publicado en prensa o redes que os parezca engañoso, y cómo lo corregiríais.